This notebook should be used to mask irrelevant parts in the video (outside of maze borders) before running DLC analysis in order to avoid wrong identifications of bodypart as much as possible.

In [2]:
import cv2
import os
import numpy as np

In [5]:
# Specify the directory containing video files
video_directory = "/var/home/almogmeir/Documents/M.Sc/Project/Input/videos"

# Load the list of video files as an array
video_files = [f for f in os.listdir(video_directory) if f.endswith(('.mp4', '.avi'))]
print(video_files)

['12NPM_03_12_25_2025_03_12_10_58_38_cam1.avi', '12NPM_03_12_25_2025_03_12_10_58_38_cam1_masked_masked.mp4']


In [9]:
# Define a function to capture mask coordinates with a brush-like tool
mask = None
brush_radius = 15
drawing = False  # True if the mouse is pressed

def draw_with_brush(event, x, y, flags, param):
    global drawing, mask, frame_copy
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            # Draw a circle on the mask
            # mask = cv2.resize(mask, (984, 768))
            cv2.circle(mask, (x, y), brush_radius, (255, 255, 255), -1)
            # Apply the mask with opacity to the frame copy
            frame_copy = cv2.resize(frame_copy, (984, 768))
            frame_copy = cv2.addWeighted(frame, 0.5, mask, 0.5, 0)
            cv2.imshow("Set Mask Areas", frame_copy)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False

# Load the first video file
video_path = os.path.join(video_directory, video_files[0])
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    ret, frame = cap.read()
    frame = cv2.resize(frame, (984, 768))
    if ret:
        # Initialize the mask and frame copy
        mask = np.zeros_like(frame, dtype=np.uint8)
        # mask = cv2.resize(mask, (984, 768))
        frame_copy = frame.copy()
        # resize the frame for display
        # frame_copy = cv2.resize(frame_copy, (984, 768))
        cv2.namedWindow("Set Mask Areas")
        cv2.resizeWindow("Set Mask Areas", 984, 768)
        # cv2.setWindowProperty("Set Mask Areas", cv2.WND_PROP_AUTOSIZE, cv2.WINDOW_AUTOSIZE)        
        cv2.setMouseCallback("Set Mask Areas", draw_with_brush)
        
        print("Draw with the brush to set the mask area. Press 'q' to quit.")
        while True:
            cv2.imshow("Set Mask Areas", frame_copy)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
        
        cv2.destroyAllWindows()
        print("Masking completed.")
    else:
        print("Error: Could not read the first frame.")

cap.release()


Draw with the brush to set the mask area. Press 'q' to quit.
Masking completed.


In [12]:
import os
import cv2


# Ensure video_path is defined appropriately
print("Using video file:", video_path)

# Open the video file and verify it opened correctly.
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: Cannot open video file:", video_path)
    exit()

# Retrieve video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# Resize the mask back to its original size
mask = cv2.resize(mask, (frame_width, frame_height))
fps = cap.get(cv2.CAP_PROP_FPS)
print("Video properties - Width:", frame_width, "Height:", frame_height, "FPS:", fps)

if frame_width <= 0 or frame_height <= 0:
    print("Error: Invalid frame dimensions.")
    cap.release()
    exit()

# Define the output video path, ensure its directory exists
output_video_path = video_path.replace('.avi', '_masked.mp4').replace('.mp4', '_masked.mp4')
output_dir = os.path.dirname(os.path.abspath(output_video_path))
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
print("Output video will be saved in:", output_dir)

# Use mp4v codec for the MP4 container
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
if not out.isOpened():
    print("Error: VideoWriter not opened. Check the codec and output path.")
    cap.release()
    exit()

# Process video frames
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Ensure that mask is defined and matches frame dimensions.
    # Invert the mask if required by your application.
    inverted_mask = cv2.bitwise_not(mask)
    masked_frame = cv2.bitwise_and(frame, inverted_mask)
    
    out.write(masked_frame)

# Clean up resources
cap.release()
out.release()

if os.path.exists(output_video_path):
    print(f"Masked video saved to: {output_video_path}")
else:
    print("Error: Output video file was not created.")

Using video file: /var/home/almogmeir/Documents/M.Sc/Project/Input/videos/12NPM_03_12_25_2025_03_12_10_58_38_cam1.avi
Video properties - Width: 1312 Height: 1024 FPS: 40.0
Output video will be saved in: /var/home/almogmeir/Documents/M.Sc/Project/Input/videos
Masked video saved to: /var/home/almogmeir/Documents/M.Sc/Project/Input/videos/12NPM_03_12_25_2025_03_12_10_58_38_cam1_masked_masked.mp4


In [ ]:
# Show the masked video first frame
cap = cv2.VideoCapture(output_video_path)
if not cap.isOpened():
    print("Error: Cannot open masked video file:", output_video_path)
    exit()
ret, frame = cap.read()
if ret:
    cv2.imshow("Masked Video First Frame", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Error: Could not read the first frame of the masked video.")
# Release the video capture
cap.release()

# Clean up any remaining windows
cv2.destroyAllWindows()
# Note: Ensure that the OpenCV library is installed in your Python environment.

NameError: name 'cv2' is not defined